# Detecting Fake News Beyond Source Cues: TF-IDF vs. Pretrained Sentence Embeddings
**NLP – Section 3 · Group 7 · Option 4 (student-proposed project)**
Arwa Alqahtani (2230007398) · Jana Algahtani (2230003732) · Mashael Alqahtani (2230006753)

**How to run:** open in Google Colab, choose *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
No files need to be uploaded: the ISOT data are downloaded from Kaggle in the Data section. A full run takes about 15 minutes (most of it is the MiniLM embeddings and the TF-IDF grid).

| Section | Owner |
|---|---|
| Setup, Data | Arwa |
| Method 1: TF-IDF + Logistic Regression | Jana |
| Method 2: MiniLM sentence embeddings + Logistic Regression | Mashael |
| Results and error analysis | Jana, Mashael |
| Try it, original examples | all |
| Sources | all |

## Setup

In [ ]:
!pip install -q sentence-transformers

import os, re, sys, random, zipfile, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn, joblib, torch
import sentence_transformers
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42                      # one seed for every random step in the project
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR  = Path("data")       # relative paths only
SPLIT_DIR = DATA_DIR / "splits"
MODEL_DIR = Path("models")
RES_DIR   = Path("results")
for d in (DATA_DIR, SPLIT_DIR, MODEL_DIR, RES_DIR):
    d.mkdir(parents=True, exist_ok=True)

N_WORDS = 200                  # body words kept in the model input

print("Python               ", sys.version.split()[0])
print("pandas               ", pd.__version__)
print("numpy                ", np.__version__)
print("scikit-learn         ", sklearn.__version__)
print("torch                ", torch.__version__)
print("sentence-transformers", sentence_transformers.__version__)
print("GPU                  ", torch.cuda.is_available())

## Data
**Owner:** Arwa Alqahtani (Team Leader / Data)

This part of the notebook:
1. loads the ISOT Fake News Dataset (`Fake.csv`, `True.csv`) and gives **Fake = 1, Real = 0**,
2. removes empty articles and duplicates **before** splitting,
3. builds the model input (**title + first 200 words of the body**) in two versions: `input` (original) and `input_clean` (source cues removed),
4. makes a stratified **70 / 15 / 15** train / dev / test split with a fixed seed and saves `train.csv`, `dev.csv`, `test.csv`,
5. checks that no example appears in more than one split.

The `subject` and `date` columns are **not used**: their values differ completely between the two classes, so they would leak the label.

### Load the dataset
The notebook finds the data in this order, so **Run all works from a fresh session**:
1. `data/Fake.csv` and `data/True.csv` already there → used directly;
2. otherwise a Kaggle `.zip` uploaded to the session (or to `data/`) → unzipped into `data/`;
3. otherwise downloaded from Kaggle with `kagglehub` (pre-installed on Colab; public datasets need no login).

The row counts are checked, so every run uses exactly the same files.
Note: the Kaggle data card says 23,502 fake articles, but `Fake.csv` actually has **23,481** rows; we report the real file counts.

In [ ]:
KAGGLE_DATASET = "clmentbisaillon/fake-and-real-news-dataset"   # ISOT Fake.csv / True.csv on Kaggle
EXPECTED_ROWS = {"Fake.csv": 23481, "True.csv": 21417}          # row counts of the ISOT files

fake_path, true_path = DATA_DIR / "Fake.csv", DATA_DIR / "True.csv"

def have_data():
    return fake_path.exists() and true_path.exists()

# 2) a Kaggle zip uploaded by hand (e.g. "archive (13).zip")
if not have_data():
    for z in list(Path(".").glob("*.zip")) + list(DATA_DIR.glob("*.zip")):
        with zipfile.ZipFile(z) as zf:
            names = {Path(n).name: n for n in zf.namelist()}
            if "Fake.csv" in names and "True.csv" in names:
                for name in ["Fake.csv", "True.csv"]:
                    with zf.open(names[name]) as src_f, open(DATA_DIR / name, "wb") as dst_f:
                        shutil.copyfileobj(src_f, dst_f)
                print("Unzipped", z)
                break

# 3) download from Kaggle
if not have_data():
    import kagglehub
    src = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    for name in ["Fake.csv", "True.csv"]:
        found = next(src.rglob(name))          # file may sit in a sub-folder
        shutil.copy(found, DATA_DIR / name)
    print("Downloaded from Kaggle:", KAGGLE_DATASET)

fake = pd.read_csv(fake_path)
real = pd.read_csv(true_path)
fake["label"] = 1      # Fake = 1
real["label"] = 0      # Real = 0

# a stable id that points back to the original file and row
fake["id"] = ["fake_" + str(i) for i in range(len(fake))]
real["id"] = ["real_" + str(i) for i in range(len(real))]

raw = pd.concat([fake, real], ignore_index=True)
raw = raw[["id", "title", "text", "label"]]       # subject and date dropped on purpose (label leakage)

assert len(fake) == EXPECTED_ROWS["Fake.csv"] and len(real) == EXPECTED_ROWS["True.csv"],     f"Unexpected files: Fake={len(fake)}, True={len(real)}"
print("Fake.csv rows:", len(fake))
print("True.csv rows:", len(real))
print("Total rows   :", len(raw))

### Source-cue removal: `remove_cues`
In ISOT, real articles carry Reuters cues and fake articles carry their own artefacts.
A model could score highly by learning these cues instead of the content.
`remove_cues` removes:

| Cue | Mostly in | Example |
|---|---|---|
| Reuters dateline at the start | Real | `WASHINGTON (Reuters) - ` |
| The word *Reuters* anywhere else | Real | `told Reuters`, `Thomson Reuters` |
| URLs and Twitter picture links | Fake | `https://t.co/...`, `pic.twitter.com/...` |
| Twitter handles | Fake | `@realDonaldTrump` |
| Image-credit phrases | Fake | `Featured image via Getty Images`, `Photo by ...` |
| Reuters boilerplate | Real | `The following statements were posted to the verified Twitter accounts of ... Reuters has not edited the statements ...`, `(Reporting by ...; Editing by ...)` |
| Site names | Fake | `21st Century Wire`, `21WIRE` |
| Media tags in titles | Fake | `(VIDEO)`, `[TWEETS]`, `(WATCH)` |

The rules are fixed by hand (nothing is learned from the data), so applying them before the split does not leak test information.

In [ ]:
_DATELINE   = re.compile(r"^[^\n]{0,150}?\(\s*reuters\s*\)\s*[-\u2013\u2014]*\s*", re.I)
_URL        = re.compile(r"(https?://\S+|www\.\S+|pic\.twitter\.com/\S+|\bbit\.ly/\S+|\bt\.co/\S+)", re.I)
_HANDLE     = re.compile(r"@\w+")
_IMG_CREDIT = re.compile(r"(featured image|header image|images?|photos?|screenshot|screen capture|screengrab)"
                         r"\s*(via|by|credit|credits|courtesy of|from)\b[^.\n]*", re.I)
_SITE_NAMES = re.compile(r"getty\s*images|\b(?:21st century wire|21wire|reuters)\b", re.I)
_MEDIA_TAG  = re.compile(r"[\(\[]\s*(video|videos|tweet|tweets|watch|image|images|photo|photos|audio|"
                         r"screenshot|screenshots|details|graphic video)\s*[\)\]]", re.I)
# Reuters note on "Trump on Twitter" articles; matched in 3 parts with \s+ so extra or
# non-breaking spaces between words do not stop the match
_TWEET_NOTE = [
    re.compile(r"the\s+following\s+statements?\s+(?:was|were)\s+posted\s+to\s+the\s+verified\s+"
               r"twitter\s+accounts?\s+of\s+(?:u\.\s*s\.\s+)?(?:president(?:-elect)?\s+)?donald\s+(?:j\.\s+)?trump"
               r"(?:\s*,?\s*(?:and\s+)?@\w+)*\s*\.?", re.I),
    re.compile(r"the\s+opinions?\s+expressed\s+(?:are|is)\s+(?:his|her|their)\s+own\s*\.?", re.I),
    re.compile(r"(?:reuters\s+)?has\s+not\s+edited\s+the\s+statements?\s+or\s+confirmed\s+(?:their|its)\s+accuracy\s*\.?", re.I),
]
_CREDITS    = re.compile(r"\(\s*(?:additional\s+)?(?:reporting|writing|editing|compiled|this story|this version)\b[^)]*\)", re.I)
_EMPTY_PAR  = re.compile(r"\(\s*\)|\[\s*\]")
_SPACES     = re.compile(r"\s+")

def remove_cues(text: str) -> str:
    """Remove source-identifying cues from a title or article body.
    Repeated until nothing changes, because removing one cue can bring two
    parts of another cue together, e.g. "Image (VIDEO) via YouTube"."""
    if not isinstance(text, str):
        return ""
    t = _DATELINE.sub("", text)
    for _ in range(5):
        before = t
        for rx in _TWEET_NOTE:
            t = rx.sub(" ", t)
        t = _CREDITS.sub(" ", t)
        t = _URL.sub(" ", t)
        t = _HANDLE.sub(" ", t)
        t = _IMG_CREDIT.sub(" ", t)
        t = _MEDIA_TAG.sub(" ", t)
        t = _SITE_NAMES.sub(" ", t)
        t = _EMPTY_PAR.sub(" ", t)
        t = _SPACES.sub(" ", t)
        if t == before:
            break
    return _SPACES.sub(" ", t).strip()

def first_words(text: str, n: int = N_WORDS) -> str:
    return " ".join(str(text).split()[:n])

def make_input(title: str, body: str) -> str:
    """Model input = title + first N_WORDS words of the body (same for both methods)."""
    return (str(title).strip() + ". " + first_words(body)).strip()

# quick checks that the rules do what we expect
tests = {
    "WASHINGTON (Reuters) - The Senate voted on Tuesday.": "The Senate voted on Tuesday.",
    "LONDON/PARIS (Reuters) -  Talks resumed.": "Talks resumed.",
    "He told Reuters the plan was final.": "He told the plan was final.",
    "Watch here https://t.co/abc123 pic.twitter.com/XyZ now": "Watch here now",
    "— Donald J. Trump (@realDonaldTrump) May 5, 2017": "— Donald J. Trump May 5, 2017",
    "The end. Featured image via Getty Images": "The end.",
    "Photo by Chip Somodevilla/Getty Images. Next sentence.": ". Next sentence.",
    "Trump SLAMS reporter (VIDEO)": "Trump SLAMS reporter",
    "21st Century Wire says the vote was rigged": "says the vote was rigged",
    ("The following statements were posted to the verified Twitter accounts of U.S. President Donald Trump, "
     "@realDonaldTrump and @POTUS. The opinions expressed are his own. Reuters has not edited the statements "
     "or confirmed their accuracy. @realDonaldTrump : - Tax cuts"): ": - Tax cuts",
    "Prices rose. (Reporting by Jane Doe; Editing by Tom Brown)": "Prices rose.",
    ("The following  statements were posted to the verified Twitter accounts of U.S. President Donald Trump, "
     "@realDonaldTrump and @POTUS.  The opinions expressed are his own. Reuters has not edited the  statements "
     "or confirmed their accuracy.  @realDonaldTrump : - Tax cuts"): ": - Tax cuts",
    "The Federal Reserve kept rates unchanged.": "The Federal Reserve kept rates unchanged.",
}
for raw_text, expected in tests.items():
    got = remove_cues(raw_text)
    assert got == expected, f"\n input:    {raw_text!r}\n expected: {expected!r}\n got:      {got!r}"
print(f"All {len(tests)} remove_cues checks passed.")

### Build the model input and drop empty articles
`input` = title + first 200 words of the original body.
`input_clean` = cues removed from the title and the **full** body first, then the first 200 words are taken, so the clean version still has up to 200 words of content.

In [ ]:
df = raw.copy()
df["title"] = df["title"].fillna("").astype(str)
df["text"]  = df["text"].fillna("").astype(str)

n0 = len(df)
empty_body = df["text"].str.strip() == ""
print("Rows with an empty body (dropped):", int(empty_body.sum()), "->", df.loc[empty_body, "label"].map({1: "Fake", 0: "Real"}).value_counts().to_dict())
df = df[~empty_body].copy()

df["input"]       = [make_input(t, b) for t, b in zip(df["title"], df["text"])]
df["clean_title"] = df["title"].map(remove_cues)
df["clean_body"]  = df["text"].map(remove_cues)
df["input_clean"] = [make_input(t, b) for t, b in zip(df["clean_title"], df["clean_body"])]

# an article whose body was only links / credits has nothing left to classify
empty_clean = df["clean_body"].str.strip() == ""
print("Rows whose body is empty after cue removal (dropped):", int(empty_clean.sum()))
df = df[~empty_clean].copy()
print("Rows kept:", len(df), "of", n0)

### Remove duplicates before splitting
Duplicates are found on normalised text (lowercase, single spaces), in two passes:
1. same article body,
2. same **clean model input** — two articles that differ only in a cue (e.g. the dateline) or only after word 200 would give the model the same input, so only one is kept.

If the same text appears with **both** labels, all its copies are removed, because the label is unreliable.

In [ ]:
def norm(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).lower()).strip()

def dedup(frame: pd.DataFrame, key_col: str, step: str) -> pd.DataFrame:
    key = frame[key_col].map(norm)
    n_labels = frame.groupby(key)["label"].transform("nunique")
    conflict = n_labels > 1
    out = frame[~conflict]
    out = out[~key[~conflict].duplicated(keep="first")]
    print(f"{step}: removed {int(conflict.sum())} conflicting-label rows and "
          f"{len(frame) - int(conflict.sum()) - len(out)} duplicates -> {len(out)} rows")
    return out

before = df["label"].value_counts()
df = dedup(df, "text", "Pass 1 (same body)")
df = dedup(df, "input_clean", "Pass 2 (same clean input)")
after = df["label"].value_counts()

dedup_table = pd.DataFrame({"before": before, "after": after}).rename(index={1: "Fake (1)", 0: "Real (0)"})
dedup_table.loc["Total"] = dedup_table.sum()
dedup_table

### Stratified 70 / 15 / 15 split and save
Split once into train (70%) and a temporary 30%, then split the 30% in half into dev and test. Both splits are stratified by label and use `SEED`.

In [ ]:
cols = ["id", "label", "title", "input", "input_clean"]
data = df[cols].reset_index(drop=True)

train, temp = train_test_split(data, test_size=0.30, stratify=data["label"], random_state=SEED)
dev, test   = train_test_split(temp, test_size=0.50, stratify=temp["label"], random_state=SEED)

for name, part in [("train", train), ("dev", dev), ("test", test)]:
    part.sort_values("id").to_csv(SPLIT_DIR / f"{name}.csv", index=False)

split_table = pd.DataFrame({
    name: {"Fake (1)": int((p.label == 1).sum()), "Real (0)": int((p.label == 0).sum()), "Total": len(p),
           "Share of data": f"{len(p) / len(data):.1%}", "Fake share": f"{p.label.mean():.1%}"}
    for name, p in [("train", train), ("dev", dev), ("test", test)]
}).T
print("Saved to", SPLIT_DIR.resolve())
split_table

### Checks: no overlap between splits, and cues really removed

In [ ]:
# 1) no example (by id or by clean input) appears in two splits
for a_name, a, b_name, b in [("train", train, "dev", dev), ("train", train, "test", test), ("dev", dev, "test", test)]:
    assert not set(a.id) & set(b.id), f"id overlap {a_name}/{b_name}"
    assert not set(a.input_clean.map(norm)) & set(b.input_clean.map(norm)), f"text overlap {a_name}/{b_name}"
print("No id or text overlap between train, dev and test.")

# 2) how often each cue appears before and after cleaning, per class
cue_patterns = {
    "Reuters":            r"\breuters\b",
    "URL / pic.twitter":  r"https?://|www\.|pic\.twitter\.com",
    "@handle":            r"@\w+",
    "Image credit":       r"(?:featured image|photo|image)\s*(?:via|by)|getty images",
    "Media tag (VIDEO)":  r"[\(\[]\s*(?:video|tweets?|watch)\s*[\)\]]",
    "Reuters boilerplate": r"verified twitter accounts? of|has not edited the statements|\((?:additional )?(?:reporting|editing) by",
}
rows = []
for cue, pat in cue_patterns.items():
    row = {"Cue": cue}
    for lab, lab_name in [(0, "Real"), (1, "Fake")]:
        part = data[data.label == lab]
        row[f"{lab_name} original"] = f"{part.input.str.contains(pat, case=False, regex=True).mean():.1%}"
        row[f"{lab_name} clean"]    = f"{part.input_clean.str.contains(pat, case=False, regex=True).mean():.1%}"
    rows.append(row)
cue_table = pd.DataFrame(rows).set_index("Cue")
left = {cue: int(data.input_clean.str.contains(pat, case=False, regex=True).sum()) for cue, pat in cue_patterns.items()}
assert sum(left.values()) == 0, f"Cues left after cleaning: {left}"
print("Every cue is at 0 in input_clean.")
cue_table      # % of articles in each class whose model input contains the cue

### Before / after examples

In [ ]:
pd.set_option("display.max_colwidth", 300)
examples = pd.concat([data[data.label == 0].sample(2, random_state=SEED),
                      data[data.label == 1].sample(2, random_state=SEED)])
for _, r in examples.iterrows():
    print(f"[{r.id}]  label = {'Fake' if r.label == 1 else 'Real'}")
    print("  input      :", r.input[:250], "...")
    print("  input_clean:", r.input_clean[:250], "...\n")

### Load the saved splits
Both methods read the saved `train.csv`, `dev.csv` and `test.csv`, so they use exactly the same examples in the same order.

In [ ]:
train = pd.read_csv(SPLIT_DIR / "train.csv")
dev   = pd.read_csv(SPLIT_DIR / "dev.csv")
test  = pd.read_csv(SPLIT_DIR / "test.csv")
y_train, y_dev, y_test = train["label"].values, dev["label"].values, test["label"].values

for name, part in [("train", train), ("dev", dev), ("test", test)]:
    print(f"{name:5s} {len(part):6d} rows | Fake share {part.label.mean():.1%}")

## Method 1 (traditional): TF-IDF + Logistic Regression
**Owner:** Jana Algahtani. `ngram_range` and `C` are tuned on the **dev** set only (macro F1); the test set is used once, in the Results section.

In [ ]:
X_train_m1, X_dev_m1 = train["input"], dev["input"]

### Tune `ngram_range` and `C` on the dev set
TF-IDF (lowercase, terms in at least 2 documents, sublinear tf) + Logistic Regression.
Grid: `ngram_range` ∈ {(1,1), (1,2)} × `C` ∈ {0.1, 1, 10, 100, 1000, 10⁴, 10⁵}. Selected by **dev macro F1**; on a tie the smaller C is kept.
C = 10⁴ and 10⁵ were added to check that the best C is not at the edge of the grid.

In [ ]:
def make_model(ngram, C):
    return Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=ngram,
                                  min_df=2, sublinear_tf=True)),
        ("lr", LogisticRegression(C=C, max_iter=2000, random_state=SEED)),
    ])

results = []
for ngram in [(1, 1), (1, 2)]:
    for C in [0.1, 1, 10, 100, 1000, 10000, 100000]:
        pred = make_model(ngram, C).fit(X_train_m1, y_train).predict(X_dev_m1)
        results.append({"ngram": str(ngram), "C": C,
                        "dev_acc": accuracy_score(y_dev, pred),
                        "dev_macro_f1": f1_score(y_dev, pred, average="macro")})
        print(f"ngram={ngram}  C={C:<6}  dev macro F1={results[-1]['dev_macro_f1']:.4f}")

# ties -> the smaller C (simpler model) wins
results = (pd.DataFrame(results)
           .sort_values(["dev_macro_f1", "C"], ascending=[False, True], kind="mergesort")
           .reset_index(drop=True))
results.round(4)

### Save the best model (trained on `input`)

In [ ]:
best = results.iloc[0]
BEST_NGRAM, BEST_C = eval(best["ngram"]), float(best["C"])
print(f"Best: ngram_range={BEST_NGRAM}, C={BEST_C}, dev macro F1={best['dev_macro_f1']:.4f}")

best_model = make_model(BEST_NGRAM, BEST_C).fit(X_train_m1, y_train)
joblib.dump(best_model, MODEL_DIR / "method1_tfidf_lr.joblib")
print("Vocabulary size:", len(best_model.named_steps["tfidf"].vocabulary_))

### Final model on cue-removed text (used by Try it)
`ngram_range` and `C` are tuned again, this time on `input_clean` (train → dev), the same way as Method 2's final model.

In [ ]:
X_train_c, X_dev_c = train["input_clean"], dev["input_clean"]

results_clean = []
for ngram in [(1, 1), (1, 2)]:
    for C in [0.1, 1, 10, 100, 1000, 10000, 100000]:
        pred = make_model(ngram, C).fit(X_train_c, y_train).predict(X_dev_c)
        results_clean.append({"ngram": str(ngram), "C": C,
                              "dev_acc": accuracy_score(y_dev, pred),
                              "dev_macro_f1": f1_score(y_dev, pred, average="macro")})
        print(f"ngram={ngram}  C={C:<6}  dev macro F1 (clean)={results_clean[-1]['dev_macro_f1']:.4f}")

# ties -> the smaller C (simpler model) wins
results_clean = (pd.DataFrame(results_clean)
                 .sort_values(["dev_macro_f1", "C"], ascending=[False, True], kind="mergesort")
                 .reset_index(drop=True))
best_c = results_clean.iloc[0]
CLEAN_NGRAM, CLEAN_C = eval(best_c["ngram"]), float(best_c["C"])
print(f"\nBest on clean text: ngram_range={CLEAN_NGRAM}, C={CLEAN_C}, dev macro F1={best_c['dev_macro_f1']:.4f}")

final_model = make_model(CLEAN_NGRAM, CLEAN_C).fit(X_train_c, y_train)
joblib.dump(final_model, MODEL_DIR / "method1_final_clean.joblib")
results_clean.round(4)

## Method 2 (pretrained): MiniLM sentence embeddings + Logistic Regression
**Owner:** Mashael Alqahtani. The pretrained all-MiniLM-L6-v2 model is used as a frozen feature extractor; only the logistic regression is trained.

### Sentence embeddings
Each text is converted by the pretrained all-MiniLM-L6-v2 model into 384 numbers. Results are saved as .npy files so they are computed only once.

In [ ]:
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

EMB_DIR = Path("data/embeddings")
EMB_DIR.mkdir(parents=True, exist_ok=True)

model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

def get_embeddings(texts, name):
    """Encode texts with MiniLM; load from disk if already computed."""
    path = EMB_DIR / f"{name}.npy"
    if path.exists():
        print(f"{name}: loaded from {path}")
        return np.load(path)
    X = model.encode(list(texts), batch_size=128, show_progress_bar=True)
    np.save(path, X)
    print(f"{name}: computed {X.shape}")
    return X

# Original text (training + tuning + first evaluation)
X_train = get_embeddings(train["input"], "train_input")
X_dev   = get_embeddings(dev["input"],   "dev_input")
X_test  = get_embeddings(test["input"],  "test_input")

# Cue-removed text
X_test_clean  = get_embeddings(test["input_clean"],  "test_clean")
X_train_clean = get_embeddings(train["input_clean"], "train_clean")
X_dev_clean   = get_embeddings(dev["input_clean"],   "dev_clean")

### Tuning C and testing
C is chosen on the dev set only. The model is then tested once on the same test examples, original and cue-removed.

In [ ]:
C_GRID = [0.01, 0.1, 1, 3, 10, 30, 100, 300, 1000]

def tune_C(X_tr, y_tr, X_dv, y_dv):
    """Pick C with the highest macro F1 on the dev set."""
    scores = {}
    for C in C_GRID:
        clf = LogisticRegression(C=C, max_iter=2000, random_state=SEED)
        clf.fit(X_tr, y_tr)
        scores[C] = f1_score(y_dv, clf.predict(X_dv), average="macro")
        print(f"C={C:<6} dev macro F1 = {scores[C]:.4f}")
    best_C = max(scores, key=scores.get)
    print("Best C:", best_C)
    return best_C

def evaluate(y_true, y_pred):
    return {"accuracy": round(accuracy_score(y_true, y_pred), 4),
            "macro_F1": round(f1_score(y_true, y_pred, average="macro"), 4)}

# 1) Tune on dev (original text)
best_C = tune_C(X_train, y_train, X_dev, y_dev)

# 2) Train once with best C, test on the SAME examples: original vs cue-removed
m2 = LogisticRegression(C=best_C, max_iter=2000, random_state=SEED)
m2.fit(X_train, y_train)

pred_orig  = m2.predict(X_test)
pred_clean = m2.predict(X_test_clean)

results_m2 = pd.DataFrame({
    "Original test":    evaluate(y_test, pred_orig),
    "Cue-removed test": evaluate(y_test, pred_clean),
}).T
results_m2.loc["Drop"] = results_m2.loc["Original test"] - results_m2.loc["Cue-removed test"]
results_m2.index.name = "Method 2: MiniLM + LR"

results_m2.to_csv("results/method2_results.csv")
print()
print(results_m2)
print()
print(classification_report(y_test, pred_clean, target_names=["Real", "Fake"], digits=4))

### Final model on cue-removed text (used by Try it)
Retrained on cue-removed text and saved, so new text can be classified without retraining.

In [ ]:
# 1) Tune C again, this time on cue-removed text
print("Tuning on cue-removed text:")
best_C_clean = tune_C(X_train_clean, y_train, X_dev_clean, y_dev)

# 2) Final model: trained on cue-removed text
m2_final = LogisticRegression(C=best_C_clean, max_iter=2000, random_state=SEED)
m2_final.fit(X_train_clean, y_train)

final_scores = evaluate(y_test, m2_final.predict(X_test_clean))
print("\nFinal model (trained on clean) on cue-removed test:", final_scores)

# 3) Save it so the Try-it cell needs no retraining
joblib.dump(m2_final, "models/method2_minilm_lr.joblib")
print("Saved: models/method2_minilm_lr.joblib")

## Results on the same test examples
Both methods were trained on the original text and are tested once on the **same 5,787 test examples**, original and cue-removed.
The Try-it models (retrained on cue-removed text) are also scored on the cue-removed test set.

In [ ]:
def scores(y_true, y_pred):
    return accuracy_score(y_true, y_pred), f1_score(y_true, y_pred, average="macro")

rows = []
for method, pred_o, pred_c in [
    ("TF-IDF + LR", best_model.predict(test["input"]), best_model.predict(test["input_clean"])),
    ("MiniLM + LR", pred_orig, pred_clean),
]:
    (acc_o, f1_o), (acc_c, f1_c) = scores(y_test, pred_o), scores(y_test, pred_c)
    rows += [{"Method": method, "Test text": "Original",    "Accuracy": acc_o, "Macro F1": f1_o},
             {"Method": method, "Test text": "Cue-removed", "Accuracy": acc_c, "Macro F1": f1_c},
             {"Method": method, "Test text": "Drop",        "Accuracy": acc_o - acc_c, "Macro F1": f1_o - f1_c}]
comparison = pd.DataFrame(rows)
comparison.to_csv(RES_DIR / "comparison.csv", index=False)
print("Both methods trained on original text:")
display(comparison.round(4))

tryit_rows = []
for method, pred in [("TF-IDF + LR (Try-it model)", final_model.predict(test["input_clean"])),
                     ("MiniLM + LR (Try-it model)", m2_final.predict(X_test_clean))]:
    acc, f1 = scores(y_test, pred)
    tryit_rows.append({"Model trained on cue-removed text": method, "Accuracy": acc, "Macro F1": f1})
print("Try-it models on the cue-removed test set:")
display(pd.DataFrame(tryit_rows).round(4))

### What Method 1 relies on
The largest logistic-regression weights of the TF-IDF model (trained on original text). Positive weights push towards Fake, negative towards Real.

In [ ]:
vocab = np.array(best_model.named_steps["tfidf"].get_feature_names_out())
coef  = best_model.named_steps["lr"].coef_[0]
order = np.argsort(coef)
print("Strongest Real features:", ", ".join(vocab[order[:15]]))
print("Strongest Fake features:", ", ".join(vocab[order[-15:][::-1]]))

### Error analysis (Method 2, cue-removed test)
Wrong predictions on the cue-removed test set, sorted by model confidence.

In [ ]:
label_name = {0: "Real", 1: "Fake"}
prob_fake = m2.predict_proba(X_test_clean)[:, 1]

err = test.copy()
err["predicted"] = pred_clean
err["prob_fake"] = prob_fake
err["confidence"] = np.where(pred_clean == 1, prob_fake, 1 - prob_fake)
err = err[err["predicted"] != err["label"]]

print("Errors:", len(err), "of", len(test), f"({len(err)/len(test):.1%})")
print("Real predicted as Fake:", int((err.label == 0).sum()))
print("Fake predicted as Real:", int((err.label == 1).sum()))

err.sort_values("confidence", ascending=False).to_csv("results/method2_errors.csv", index=False)

def show(rows, title):
    print(f"\n========== {title} ==========")
    for _, r in rows.iterrows():
        print(f"\n[{r.id}] True: {label_name[r.label]} | Predicted: {label_name[r.predicted]} "
              f"| P(fake) = {r.prob_fake:.2f}")
        print("  ", r.input_clean[:300], "...")

top = err.sort_values("confidence", ascending=False)
show(top[top.label == 0].head(3), "Most confident: Real predicted as Fake")
show(top[top.label == 1].head(3), "Most confident: Fake predicted as Real")

## Try it
Both final models were trained on cue-removed text and saved, so new text is classified **without retraining**.
The user's text goes through `remove_cues()` first. Empty input gives a message instead of a prediction.

In [ ]:
tryit_model = joblib.load(MODEL_DIR / "method1_final_clean.joblib")
m2_final    = joblib.load(MODEL_DIR / "method2_minilm_lr.joblib")

def predict_m1(text):
    """Method 1 (TF-IDF + LR) on new text."""
    if text is None or not str(text).strip():
        return "Please enter a news text (it was empty)."
    clean = remove_cues(str(text))
    if not clean:
        return "Nothing left to classify after removing links and source tags."
    p_fake = tryit_model.predict_proba([clean])[0, 1]
    return f"{'Fake' if p_fake >= 0.5 else 'Real'}  (P(fake) = {p_fake:.2f})"

def predict_m2(text, title=""):
    """Method 2 (MiniLM + LR) on new text."""
    if text is None or not str(text).strip():
        return "Please enter a news text (it was empty)."
    clean = make_input(remove_cues(title), remove_cues(text))
    p_fake = m2_final.predict_proba(model.encode([clean]))[0, 1]
    return f"{'Fake' if p_fake >= 0.5 else 'Real'}  (P(fake) = {p_fake:.2f})"

def try_it(text):
    print("Method 1 (TF-IDF):", predict_m1(text))
    print("Method 2 (MiniLM):", predict_m2(text))

for t in ["Pope Francis endorses Donald Trump for president, shocking the world",
          "The Federal Reserve kept interest rates unchanged on Wednesday, citing steady job growth",
          "   "]:
    print(repr(t[:60]))
    try_it(t)
    print()

Type your own text:

In [ ]:
my_text = "The central bank raised interest rates by a quarter point on Thursday, citing rising inflation"
try_it(my_text)

## Original labelled examples
Each member wrote two short texts that are not taken from ISOT or any news site. They are shown separately from the test results.

In [ ]:
mashael_examples = [
    ("Real", "Saudi Arabia’s Ministry of Energy said on November 8 that an administrative building belonging to Aramco in Riyadh was the target of an attempted attack. Debris fell on the site and caused limited damage to the building, and no injuries were reported."),
    ("Fake", "BREAKING: Saudi Arabia’s “Madrasati” School Platform HACKED! Millions of Students’ Data Exposed. Insiders reveal that the Madrasati platform was secretly hacked on October 16, and officials are trying to cover it up. An unknown group is behind the attack, and nobody knows what they did with the data. You won’t believe how long this went unnoticed!"),
]

jana_examples = [
    ("Fake", "SHOCKING: Scientists Admit Drinking Coffee Before Bed Makes You Immune To The Flu, "
             "And Big Pharma Is Furious. A leaked report that the mainstream media refuses to show you "
             "proves that one cup of coffee at night completely blocks the flu virus. Doctors have known "
             "this for years but kept it secret to protect vaccine profits. Share this before it gets deleted!"),
    ("Real", "City council approves budget for new public library. The city council voted 7-2 on Tuesday "
             "to approve a $12 million budget for a new public library in the downtown area, officials said. "
             "Construction is expected to begin next spring and take about 18 months. The council said the "
             "project will be funded through a mix of state grants and existing city funds."),
]

arwa_examples = [
    ("Real", "FIFA confirmed in December 2024 that Saudi Arabia will host the 2034 FIFA World Cup, "
             "the first time the tournament will be held in the country."),
    ("Fake", "SHOCKING: Scientists CONFIRM that drinking Arabic coffee three times a day cures diabetes, "
             "and doctors don't want you to know! Share this before it gets deleted!"),
]

for author, examples in [("Arwa", arwa_examples), ("Jana", jana_examples), ("Mashael", mashael_examples)]:
    for true_label, text in examples:
        print(f"[{author}] True: {true_label}")
        print("   ", text[:120], "...")
        print("    Method 1:", predict_m1(text))
        print("    Method 2:", predict_m2(text))
        print()

## Sources
1. H. Ahmed, I. Traore and S. Saad, “Detection of online fake news using n-gram analysis and machine learning techniques,” ISDDC 2017 — the ISOT Fake News Dataset.
2. ISOT Fake News Dataset on Kaggle: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset
3. N. Reimers and I. Gurevych, “Sentence-BERT: Sentence embeddings using Siamese BERT-networks,” EMNLP-IJCNLP 2019.
4. all-MiniLM-L6-v2 (sentence-transformers, Hugging Face): https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
5. scikit-learn (TfidfVectorizer, LogisticRegression, metrics): https://scikit-learn.org
6. AI assistance: an AI assistant (Claude) was used to help write and check parts of the code (data cleaning, cue removal, merging the notebooks) and to review the report text. All results were run and checked by the team.